In [ ]:
import os
import re
import gc
import random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed
)

from arabert.preprocess import ArabertPreprocessor


SEED = 42

PERTURBATION_SEEDS = [
    42,
    43,
    44,
    45,
    46
]

SEVERITIES = [
    0.10,
    0.20,
    0.30
]

DIMENSIONS = [
    "Textual Accuracy",
    "Completeness",
    "Consistency",
    "Validity",
    "Understandability"
]


DATA_PATH = ("ADI_Dataset.csv")

TEXT_COLUMN = "text"
LABEL_COLUMN = "dialect"

MAX_LENGTH = 128

LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
NUM_EPOCHS = 3
WEIGHT_DECAY = 0.01


OUTPUT_DIR = (
    "/content/drive/MyDrive/ISO_Check/ADI/"
    "ALL_5_PERTURBATIONS"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


MODELS = {

    "AraBERTv2": {
        "model_name":
            "aubmindlab/bert-base-arabertv2",

        "arabert_preprocessing":
            True
    },

    "CAMeLBERT-Mix": {
        "model_name":
            "CAMeL-Lab/bert-base-arabic-camelbert-mix",

        "arabert_preprocessing":
            False
    }
}


def set_all_seeds(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    set_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(seed)


set_all_seeds(SEED)


print(
    "CUDA available:",
    torch.cuda.is_available()
)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


df = pd.read_csv(
    DATA_PATH
)


df = (

    df[
        [
            TEXT_COLUMN,
            LABEL_COLUMN
        ]
    ]

    .rename(
        columns={
            TEXT_COLUMN: "text",
            LABEL_COLUMN: "labels"
        }
    )

    .dropna()

    .reset_index(drop=True)
)


df["text"] = (
    df["text"]
    .astype(str)
    .str.strip()
)


df = (

    df[
        df["text"] != ""
    ]

    .reset_index(drop=True)
)


print(
    "\nDataset shape:",
    df.shape
)


print(
    "\nClass distribution:"
)

print(
    df["labels"].value_counts()
)


label_encoder = LabelEncoder()


df["labels"] = (
    label_encoder
    .fit_transform(
        df["labels"]
    )
)


num_labels = len(
    label_encoder.classes_
)


id2label = {

    i: str(label)

    for i, label
    in enumerate(
        label_encoder.classes_
    )
}


label2id = {

    label: i

    for i, label
    in id2label.items()
}


print(
    "\nNumber of classes:",
    num_labels
)


train_df, test_df = train_test_split(

    df,

    test_size=0.20,

    random_state=SEED,

    stratify=df["labels"]
)


train_df = (
    train_df
    .reset_index(drop=True)
)

test_df = (
    test_df
    .reset_index(drop=True)
)


print(
    "\nTrain:",
    train_df.shape
)

print(
    "Test:",
    test_df.shape
)


train_df.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "fixed_train_split.csv"
    ),

    index=False
)


test_df.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "fixed_test_split.csv"
    ),

    index=False
)


def stochastic_count(
    target,
    rng
):

    base = int(
        np.floor(target)
    )

    fraction = (
        target
        -
        base
    )

    if rng.random() < fraction:

        base += 1

    return base


ARABIC_PATTERN = re.compile(

    r"[\u0621-\u063A"
    r"\u0641-\u064A"
    r"\u0671-\u06D3"
    r"\u06FA-\u06FC]+"
)


def get_arabic_span(
    token,
    min_len=3
):

    matches = list(
        ARABIC_PATTERN.finditer(
            token
        )
    )


    valid = [

        match

        for match in matches

        if len(
            match.group()
        ) >= min_len
    ]


    if not valid:

        return None


    return max(

        valid,

        key=lambda x:
        len(x.group())
    )


ARABIC_CHARS = list(
    "ابتثجحخدذرزسشصضطظعغفقكلمنهوي"
    "أإآؤئءىة"
)


ORTHOGRAPHIC_ALTERNATIVES = {

    "ا": ["أ", "إ", "آ"],

    "أ": ["ا", "إ", "آ"],

    "إ": ["ا", "أ", "آ"],

    "آ": ["ا", "أ", "إ"],

    "ي": ["ى"],

    "ى": ["ي"]
}


def typo_delete(
    word,
    rng
):

    if len(word) < 2:

        return word

    i = rng.randrange(
        len(word)
    )

    return (
        word[:i]
        +
        word[i + 1:]
    )


def typo_insert(
    word,
    rng
):

    i = rng.randrange(
        len(word) + 1
    )

    c = rng.choice(
        ARABIC_CHARS
    )

    return (
        word[:i]
        +
        c
        +
        word[i:]
    )


def typo_substitute(
    word,
    rng
):

    i = rng.randrange(
        len(word)
    )

    original = word[i]


    forbidden = {
        original
    }


    forbidden.update(

        ORTHOGRAPHIC_ALTERNATIVES.get(
            original,
            []
        )
    )


    choices = [

        c

        for c in ARABIC_CHARS

        if c not in forbidden
    ]


    new_char = rng.choice(
        choices
    )


    return (
        word[:i]
        +
        new_char
        +
        word[i + 1:]
    )


def typo_transpose(
    word,
    rng
):

    candidates = [

        i

        for i in range(
            len(word) - 1
        )

        if word[i]
        != word[i + 1]
    ]


    if not candidates:

        return word


    i = rng.choice(
        candidates
    )


    chars = list(word)

    chars[i], chars[i + 1] = (
        chars[i + 1],
        chars[i]
    )


    return "".join(chars)


TYPO_FUNCTIONS = [

    typo_delete,

    typo_insert,

    typo_substitute,

    typo_transpose
]


def corrupt_word(
    word,
    rng
):

    for _ in range(20):

        func = rng.choice(
            TYPO_FUNCTIONS
        )

        result = func(
            word,
            rng
        )

        if result != word:

            return result


    return typo_insert(
        word,
        rng
    )


def perturb_accuracy(
    text,
    severity,
    rng
):

    tokens = text.split()


    eligible = [

        i

        for i, token
        in enumerate(tokens)

        if get_arabic_span(
            token
        )
        is not None
    ]


    n = len(
        eligible
    )


    if n == 0:

        return text, 0, 0


    k = stochastic_count(
        severity * n,
        rng
    )


    k = min(
        k,
        n
    )


    if k == 0:

        return text, n, 0


    selected = rng.sample(
        eligible,
        k
    )


    changed = 0


    for i in selected:

        token = tokens[i]

        match = get_arabic_span(
            token
        )


        word = match.group()


        corrupted = corrupt_word(
            word,
            rng
        )


        tokens[i] = (

            token[:match.start()]
            +
            corrupted
            +
            token[match.end():]
        )


        if corrupted != word:

            changed += 1


    return (
        " ".join(tokens),
        n,
        changed
    )


def perturb_completeness(
    text,
    severity,
    rng
):

    words = text.split()

    n = len(words)


    if n <= 1:

        return text, n, 0


    k = stochastic_count(
        severity * n,
        rng
    )


    k = min(
        k,
        n - 1
    )


    if k == 0:

        return text, n, 0


    selected = set(

        rng.sample(
            range(n),
            k
        )
    )


    remaining = [

        word

        for i, word
        in enumerate(words)

        if i not in selected
    ]


    return (
        " ".join(remaining),
        n,
        k
    )


CONSISTENCY_MAP = {

    "ا": ["أ", "إ", "آ"],

    "أ": ["ا", "إ", "آ"],

    "إ": ["ا", "أ", "آ"],

    "آ": ["ا", "أ", "إ"],

    "ي": ["ى"],

    "ى": ["ي"]
}


def perturb_consistency(
    text,
    severity,
    rng
):

    chars = list(text)


    eligible = [

        i

        for i, char
        in enumerate(chars)

        if char in CONSISTENCY_MAP
    ]


    n = len(
        eligible
    )


    if n == 0:

        return text, 0, 0


    k = stochastic_count(
        severity * n,
        rng
    )


    k = min(
        k,
        n
    )


    if k == 0:

        return text, n, 0


    selected = rng.sample(
        eligible,
        k
    )


    for i in selected:

        chars[i] = rng.choice(

            CONSISTENCY_MAP[
                chars[i]
            ]
        )


    return (
        "".join(chars),
        n,
        k
    )


INVALID_TOKENS = [

    "<dq_invalid_a>",

    "<dq_invalid_b>",

    "<dq_invalid_c>",

    "<dq_invalid_d>"
]


def perturb_validity(
    text,
    severity,
    rng
):

    words = text.split()

    n = len(words)


    if n == 0:

        return text, 0, 0


    k = stochastic_count(
        severity * n,
        rng
    )


    if k == 0:

        return text, n, 0


    result = words.copy()


    for _ in range(k):

        token = rng.choice(
            INVALID_TOKENS
        )


        position = rng.randrange(
            len(result) + 1
        )


        result.insert(
            position,
            token
        )


    return (
        " ".join(result),
        n,
        k
    )


def perturb_understandability(
    text,
    severity,
    rng
):

    words = text.split()

    n = len(words)


    if n < 2:

        return text, n, 0


    target = (
        severity * n
    )


    k = stochastic_count(
        target,
        rng
    )


    if k == 1:

        if rng.random() < 0.5:

            k = 2

        else:

            k = 0


    k = min(
        k,
        n
    )


    if k < 2:

        return text, n, 0


    selected = sorted(

        rng.sample(
            range(n),
            k
        )
    )


    selected_words = [

        words[i]

        for i in selected
    ]


    for _ in range(20):

        permuted = (
            selected_words.copy()
        )

        rng.shuffle(
            permuted
        )

        if permuted != selected_words:

            break


    else:

        permuted = (
            selected_words[1:]
            +
            selected_words[:1]
        )


    output = (
        words.copy()
    )


    for i, new_word in zip(
        selected,
        permuted
    ):

        output[i] = (
            new_word
        )


    return (
        " ".join(output),
        n,
        k
    )


PERTURBATION_FUNCTIONS = {

    "Textual Accuracy":
        perturb_accuracy,

    "Completeness":
        perturb_completeness,

    "Consistency":
        perturb_consistency,

    "Validity":
        perturb_validity,

    "Understandability":
        perturb_understandability
}


def create_perturbed_dataframe(
    clean_df,
    dimension,
    severity,
    seed
):

    rng = random.Random(
        seed
    )


    function = (
        PERTURBATION_FUNCTIONS[
            dimension
        ]
    )


    perturbed_df = (
        clean_df.copy()
    )


    new_texts = []

    total_eligible = 0

    total_changed_units = 0

    changed_instances = 0


    for text in clean_df["text"]:

        (
            new_text,
            eligible_units,
            changed_units
        ) = function(

            text,

            severity,

            rng
        )


        new_texts.append(
            new_text
        )


        total_eligible += (
            eligible_units
        )


        total_changed_units += (
            changed_units
        )


        if new_text != text:

            changed_instances += 1


    perturbed_df[
        "text"
    ] = new_texts


    assert (

        perturbed_df[
            "labels"
        ].equals(
            clean_df["labels"]
        )

    )


    realized_rate = (

        total_changed_units
        /
        total_eligible

        if total_eligible > 0

        else 0.0
    )


    changed_instance_rate = (

        changed_instances
        /
        len(clean_df)

        if len(clean_df) > 0

        else 0.0
    )


    diagnostics = {

        "Dimension":
            dimension,

        "Severity":
            severity,

        "Seed":
            seed,

        "Eligible_Units":
            total_eligible,

        "Changed_Units":
            total_changed_units,

        "Realized_Rate":
            realized_rate,

        "Changed_Instance_Rate":
            changed_instance_rate
    }


    return (
        perturbed_df,
        diagnostics
    )


def evaluate_dataset(
    trainer,
    dataset
):

    output = trainer.predict(
        dataset
    )


    predictions = np.argmax(
        output.predictions,
        axis=-1
    )


    labels = (
        output.label_ids
    )


    accuracy = accuracy_score(
        labels,
        predictions
    )


    macro_f1 = f1_score(

        labels,

        predictions,

        average="macro",

        zero_division=0
    )


    return (
        accuracy,
        macro_f1
    )


ALL_RESULTS = []

ALL_DIAGNOSTICS = []


for MODEL_LABEL, CONFIG in MODELS.items():

    print(
        "\n\n"
        +
        "#" * 80
    )

    print(
        "STARTING MODEL:",
        MODEL_LABEL
    )

    print(
        "#" * 80
    )


    set_all_seeds(
        SEED
    )


    MODEL_NAME = (
        CONFIG[
            "model_name"
        ]
    )


    if CONFIG[
        "arabert_preprocessing"
    ]:

        print(
            "\nUsing AraBERT preprocessing"
        )


        prep = ArabertPreprocessor(

            model_name=MODEL_NAME,

            keep_emojis=True
        )


        def model_preprocess(text):

            return prep.preprocess(
                str(text)
            )


    else:

        print(
            "\nUsing CAMeLBERT raw-text pipeline"
        )


        def model_preprocess(text):

            return str(text)


    train_model_df = (
        train_df.copy()
    )

    test_model_df = (
        test_df.copy()
    )


    print(
        "\nPreparing clean data..."
    )


    train_model_df[
        "text"
    ] = (

        train_model_df[
            "text"
        ]

        .apply(
            model_preprocess
        )
    )


    test_model_df[
        "text"
    ] = (

        test_model_df[
            "text"
        ]

        .apply(
            model_preprocess
        )
    )


    tokenizer = (
        AutoTokenizer
        .from_pretrained(
            MODEL_NAME
        )
    )


    model = (

        AutoModelForSequenceClassification

        .from_pretrained(

            MODEL_NAME,

            num_labels=num_labels,

            id2label=id2label,

            label2id=label2id
        )
    )


    def tokenize_function(batch):

        return tokenizer(

            batch["text"],

            truncation=True,

            max_length=MAX_LENGTH
        )


    train_dataset = (
        Dataset.from_pandas(

            train_model_df,

            preserve_index=False
        )

        .map(
            tokenize_function,
            batched=True
        )
    )


    clean_test_dataset = (
        Dataset.from_pandas(

            test_model_df,

            preserve_index=False
        )

        .map(
            tokenize_function,
            batched=True
        )
    )


    collator = (
        DataCollatorWithPadding(
            tokenizer=tokenizer
        )
    )


    training_args = TrainingArguments(

        output_dir=os.path.join(
            OUTPUT_DIR,
            MODEL_LABEL
        ),

        learning_rate=LEARNING_RATE,

        per_device_train_batch_size=(
            TRAIN_BATCH_SIZE
        ),

        per_device_eval_batch_size=(
            EVAL_BATCH_SIZE
        ),

        num_train_epochs=(
            NUM_EPOCHS
        ),

        weight_decay=(
            WEIGHT_DECAY
        ),

        logging_strategy="epoch",

        save_strategy="no",

        report_to="none",

        seed=SEED,

        data_seed=SEED,

        optim="adamw_torch"
    )


    trainer = Trainer(

        model=model,

        args=training_args,

        train_dataset=train_dataset,

        data_collator=collator
    )


    print(
        "\n========================================"
    )

    print(
        "TRAINING",
        MODEL_LABEL
    )

    print(
        "========================================"
    )


    trainer.train()


    (
        clean_accuracy,
        clean_f1
    ) = evaluate_dataset(

        trainer,

        clean_test_dataset
    )


    print(
        "\nCLEAN RESULTS"
    )

    print(
        "Accuracy:",
        round(
            clean_accuracy * 100,
            2
        )
    )

    print(
        "Macro-F1:",
        round(
            clean_f1 * 100,
            2
        )
    )


    for dimension in DIMENSIONS:

        print(
            "\n\n"
            +
            "=" * 80
        )

        print(
            MODEL_LABEL,
            "-",
            dimension
        )

        print(
            "=" * 80
        )


        for severity in SEVERITIES:

            print(
                "\nSeverity:",
                int(
                    severity * 100
                ),
                "%"
            )


            for perturb_seed in (
                PERTURBATION_SEEDS
            ):


                (
                    perturbed_df,
                    diagnostics
                ) = create_perturbed_dataframe(

                    clean_df=test_df,

                    dimension=dimension,

                    severity=severity,

                    seed=perturb_seed
                )


                model_perturbed_df = (
                    perturbed_df.copy()
                )


                model_perturbed_df[
                    "text"
                ] = (

                    model_perturbed_df[
                        "text"
                    ]

                    .apply(
                        model_preprocess
                    )
                )


                perturbed_dataset = (

                    Dataset.from_pandas(

                        model_perturbed_df,

                        preserve_index=False
                    )

                    .map(
                        tokenize_function,
                        batched=True
                    )
                )


                (
                    pert_accuracy,
                    pert_f1
                ) = evaluate_dataset(

                    trainer,

                    perturbed_dataset
                )


                delta_f1 = (
                    clean_f1
                    -
                    pert_f1
                )


                ALL_RESULTS.append({

                    "Model":
                        MODEL_LABEL,

                    "Dimension":
                        dimension,

                    "Severity":
                        int(
                            severity * 100
                        ),

                    "Seed":
                        perturb_seed,

                    "Clean_Accuracy":
                        clean_accuracy,

                    "Clean_Macro_F1":
                        clean_f1,

                    "Perturbed_Accuracy":
                        pert_accuracy,

                    "Perturbed_Macro_F1":
                        pert_f1,

                    "Delta_F1":
                        delta_f1,

                    "Realized_Rate":
                        diagnostics[
                            "Realized_Rate"
                        ],

                    "Changed_Instance_Rate":
                        diagnostics[
                            "Changed_Instance_Rate"
                        ]
                })


                diagnostics[
                    "Model"
                ] = MODEL_LABEL


                ALL_DIAGNOSTICS.append(
                    diagnostics
                )


                print(

                    f"Seed {perturb_seed}"

                    f" | F1="
                    f"{pert_f1 * 100:.2f}"

                    f" | Delta="
                    f"{delta_f1 * 100:.2f}"

                    f" | Realized="
                    f"{diagnostics['Realized_Rate'] * 100:.2f}%"

                    f" | Changed texts="
                    f"{diagnostics['Changed_Instance_Rate'] * 100:.2f}%"
                )


                del perturbed_dataset
                del model_perturbed_df
                del perturbed_df

                gc.collect()


    del trainer
    del model
    del tokenizer

    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


results_df = pd.DataFrame(
    ALL_RESULTS
)


results_df.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "all_individual_results.csv"
    ),

    index=False
)


summary_df = (

    results_df

    .groupby(
        [
            "Model",
            "Dimension",
            "Severity"
        ],

        as_index=False
    )

    .agg(

        Clean_Macro_F1=(
            "Clean_Macro_F1",
            "first"
        ),

        Macro_F1_Mean=(
            "Perturbed_Macro_F1",
            "mean"
        ),

        Macro_F1_SD=(
            "Perturbed_Macro_F1",
            "std"
        ),

        Delta_F1_Mean=(
            "Delta_F1",
            "mean"
        ),

        Delta_F1_SD=(
            "Delta_F1",
            "std"
        ),

        Accuracy_Mean=(
            "Perturbed_Accuracy",
            "mean"
        ),

        Accuracy_SD=(
            "Perturbed_Accuracy",
            "std"
        ),

        Realized_Rate_Mean=(
            "Realized_Rate",
            "mean"
        ),

        Changed_Instance_Rate_Mean=(
            "Changed_Instance_Rate",
            "mean"
        )
    )
)


percentage_columns = [

    "Clean_Macro_F1",

    "Macro_F1_Mean",

    "Macro_F1_SD",

    "Delta_F1_Mean",

    "Delta_F1_SD",

    "Accuracy_Mean",

    "Accuracy_SD",

    "Realized_Rate_Mean",

    "Changed_Instance_Rate_Mean"
]


summary_100 = (
    summary_df.copy()
)


for column in percentage_columns:

    summary_100[
        column
    ] *= 100


print(
    "\n\n"
    +
    "#" * 100
)

print(
    "FINAL RESULTS — ALL FIVE DATA-QUALITY DIMENSIONS"
)

print(
    "#" * 100
)


display(

    summary_100[
        [
            "Model",
            "Dimension",
            "Severity",
            "Clean_Macro_F1",
            "Macro_F1_Mean",
            "Macro_F1_SD",
            "Delta_F1_Mean",
            "Delta_F1_SD",
            "Realized_Rate_Mean",
            "Changed_Instance_Rate_Mean"
        ]
    ]

    .round(3)
)


paper_df = (
    summary_100.copy()
)


paper_df[
    "Macro-F1"
] = (

    paper_df[
        "Macro_F1_Mean"
    ]
    .map(
        lambda x:
        f"{x:.2f}"
    )

    +

    " ± "

    +

    paper_df[
        "Macro_F1_SD"
    ]
    .map(
        lambda x:
        f"{x:.2f}"
    )
)


paper_df[
    "Delta-F1"
] = (

    paper_df[
        "Delta_F1_Mean"
    ]
    .map(
        lambda x:
        f"{x:.2f}"
    )

    +

    " ± "

    +

    paper_df[
        "Delta_F1_SD"
    ]
    .map(
        lambda x:
        f"{x:.2f}"
    )
)


paper_df[
    "Realized Severity"
] = (

    paper_df[
        "Realized_Rate_Mean"
    ]
    .map(
        lambda x:
        f"{x:.2f}%"
    )
)


paper_table = paper_df[

    [
        "Model",
        "Dimension",
        "Severity",
        "Macro-F1",
        "Delta-F1",
        "Realized Severity"
    ]
]


print(
    "\n\n"
    +
    "=" * 100
)

print(
    "PAPER-READY TABLE"
)

print(
    "=" * 100
)


display(
    paper_table
)


dimension_ranking = (

    summary_100

    .groupby(
        [
            "Model",
            "Dimension"
        ],

        as_index=False
    )

    .agg(

        Mean_Delta_F1=(
            "Delta_F1_Mean",
            "mean"
        )
    )

    .sort_values(
        [
            "Model",
            "Mean_Delta_F1"
        ],

        ascending=[
            True,
            False
        ]
    )
)


print(
    "\n\n"
    +
    "=" * 100
)

print(
    "DATA-QUALITY SENSITIVITY RANKING"
)

print(
    "=" * 100
)


display(
    dimension_ranking.round(3)
)


cross_model = (

    summary_100

    .groupby(
        [
            "Dimension",
            "Severity"
        ],

        as_index=False
    )

    .agg(

        Mean_Delta_F1=(
            "Delta_F1_Mean",
            "mean"
        )
    )
)


print(
    "\n\n"
    +
    "=" * 100
)

print(
    "CROSS-MODEL AGGREGATE"
)

print(
    "=" * 100
)


display(
    cross_model.round(3)
)


summary_100.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "all_five_dimensions_summary.csv"
    ),

    index=False
)


paper_table.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "paper_ready_results.csv"
    ),

    index=False
)


dimension_ranking.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "dimension_ranking.csv"
    ),

    index=False
)


cross_model.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "cross_model_results.csv"
    ),

    index=False
)


diagnostics_df = pd.DataFrame(
    ALL_DIAGNOSTICS
)


diagnostics_df.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "perturbation_diagnostics.csv"
    ),

    index=False
)


print(
    "\n\nEXPERIMENT FINISHED."
)

print(
    "\nMain output:"
)

print(

    os.path.join(
        OUTPUT_DIR,
        "paper_ready_results.csv"
    )
)
